# 평가 실행 (Colab) — 검색 / 생성

**순서:** 0~2 실행 → 3에서 런타임 재시작 → 4부터 이어서.

## 규모 두 축 (헷갈리기 쉬움)
- **213** = 평가 **질문 개수** (dev 정답 쿼리 전수)
- **20만** = **코퍼스 = 검색 대상 문서 수** (질문이 답을 찾아야 할 문서 창고)

둘은 대체 관계가 아니라 **동시에 필요**하다. 코퍼스가 작으면 정답 찾기가 쉬워져 검색 점수가 부풀려진다. 20만 임베딩은 1회성 인덱싱이고, 그 뒤 213개 질문이 그 인덱스를 검색한다.

## 실행 모드
- **SMOKE=1** (4번 셀에서 설정): 질문 20개 + 코퍼스 5천 → *코드가 끝까지 도는지만* 확인. 숫자는 버린다.
- **SMOKE 해제**: 질문 213 전수 + 코퍼스 20만 → 보고용 실측.

⚠️ 2층 판정이 **유료 API**(judge=gpt-5.4-mini)다. 실측 전 반드시 SMOKE로 구조를 검증할 것.
⚠️ 자꾸 런타임이 종료되면 Colab 사용량 한도일 수 있음 → 좀 쉬었다 새 세션으로.
⚠️ 산출물은 6번 셀에서 **Drive로 백업**하고 세션을 닫을 것. `/content` 는 세션 종료 시 사라진다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 코드 받기 (강제 정렬 — 로컬 수정 있어도 pull 안 막힘)

`BRANCH` 만 바꾸면 다른 브랜치로 돌릴 수 있다. **로컬에서 커밋만 하고 push 안 한 변경은 여기 안 들어온다** — 돌리기 전에 push 됐는지 확인할 것.

In [ ]:
import os
BRANCH = 'dh'      # ← 돌릴 브랜치. 원격에 push 된 상태여야 한다.

%cd /content
if os.path.isdir('knowledge-rag-qa'):
    %cd knowledge-rag-qa
    !git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/likelion-nlp-project2/knowledge-rag-qa.git
    %cd knowledge-rag-qa
!git log --oneline -1
print('--- data/ 코퍼스 파일 (ko_miracl_reduced_corpus.jsonl 있어야 함, 약 107MB) ---')
!ls -la data/

### 1-1. 코퍼스 복사 (Drive → `data/`)

코퍼스(107MB)는 GitHub 100MB 제한으로 저장소에 없다. **Drive에 한 번 올려두고** 세션마다 이 셀로 복사한다.

- 사전 준비(1회): [drive.google.com](https://drive.google.com) 에 로컬 `data/ko_miracl_reduced_corpus.jsonl` 을 드래그해 업로드 (내 드라이브 최상단 기준. 폴더에 넣었다면 아래 경로 수정)
- 실행하면 구글 계정 인증 팝업이 뜬다 → 허용


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 내 드라이브 최상단에 올린 경우. 폴더에 넣었다면 'MyDrive/폴더명/...' 으로 수정
SRC = '/content/drive/MyDrive/ko_miracl_reduced_corpus.jsonl'
!cp "{SRC}" /content/knowledge-rag-qa/data/
!ls -lh /content/knowledge-rag-qa/data/   # 107M 짜리 jsonl 이 보이면 성공

## 2. 설치
torch/numpy는 코랩 기본 버전으로 **고정(==)**해 torchvision/torchaudio 충돌 방지.

In [ ]:
import importlib.metadata as md
pinned = f"torch=={md.version('torch')} numpy=={md.version('numpy')}"
print('고정:', pinned)
# ragas + langchain-openai: 2층 RAGAS 판정기(gpt-5.4-mini)용.
# 생성도 API(gpt-4o-mini)라 로컬 LLM을 안 올린다 → accelerate/bitsandbytes 불필요.
!pip -q install -U {pinned} datasets chromadb sentence-transformers transformers ragas langchain-openai
print('설치 완료 — 다음 셀(3) 안내대로 런타임 재시작')

## 3. ⚠️ 런타임 재시작 (필수)
메뉴 → **런타임 → 세션 다시 시작**. 그다음 **0~2번은 다시 실행하지 말고 4번부터** 이어서.
(클론된 파일·설치는 재시작해도 유지됨. 커널 메모리만 초기화.)

## 4. 검색 평가 (E1) — *생성 평가만 할 거면 건너뛰어도 된다*

**5번(생성 평가)과 독립적이다.** 5번은 `%run` 이 아니라 `import` 로 이 스크립트를 다시 불러오는데, `%run` 은 `sys.modules` 에 캐시되지 않아 **여기서 만든 인덱스를 재사용하지 못하고 처음부터 다시 임베딩한다.**

- E1 결과는 이미 `evaluation/RESULTS.md` 에 기록돼 있다 → 재측정이 목적이 아니면 이 셀은 건너뛰고 5번으로 갈 것 (GPU 약 19분 절약).
- 돌린다면: 코퍼스 약 20만 청크 임베딩 → GPU 기준 약 19분.


In [ ]:
import os
%cd /content/knowledge-rag-qa

# ── 실행 규모 ────────────────────────────────────────────────
# SMOKE=1  : 쿼리 20개 + 코퍼스 5천 → 코드가 도는지만 확인 (숫자 무의미)
# 주석 처리: 쿼리 213 전수 + 코퍼스 20만 → 보고용 실측
#   * 213=질문 수, 20만=검색 대상 문서 수. 서로 다른 축이며 둘 다 필요하다.
#     코퍼스가 작으면 정답 찾기가 쉬워져 검색 점수가 부풀려진다.
os.environ['SMOKE'] = '1'      # ← 실측할 땐 이 줄을 주석 처리

# 검색 평가는 OpenAI API 키가 필요 없다(임베딩·검색만 수행, rag/llm.py 를 import 하지 않음).
# 키는 생성 평가(5번 셀)에서만 입력받는다.
%run evaluation/run_retrieval_eval.py

## 5. 생성 평가 — 1층 규칙 기반 지표 + 2층 RAGAS(judge=gpt-5.4-mini)

### 실측(SMOKE 해제) 시 실제로 드는 비용

| 항목 | 규모 |
|---|---|
| 20만 청크 임베딩 | **2회** (원 컬렉션 + 무근거A 컬렉션) → GPU 기준 약 19분 × 2 |
| 생성 호출 (gpt-4o-mini) | **431건** = 213 × 2조건 + 무근거B 5 |
| RAGAS 판정 (gpt-5.4-mini) | answerable 중 비기권 답변만 (최대 213건) |

4번 셀을 이미 돌렸어도 임베딩은 재사용되지 않는다(`%run` 은 모듈 캐시에 안 남는다). 위 2회는 이 셀 자체의 비용이다.

⚠️ **반드시 SMOKE로 먼저 검증한 뒤** 실측할 것.

생성 답변은 판정 **전에** `result/generations_eval.jsonl` 로 저장된다 — RAGAS가 죽어도 생성분(=이미 지불한 API 비용)은 남는다.


In [ ]:
import os
%cd /content/knowledge-rag-qa

# ── RAGAS 판정기 + 생성(gpt-4o-mini) API 키 — 여기서만 필요 ──────
# 노트북에 키를 직접 적지 말 것(저장·공유 시 그대로 노출됨). getpass로 입력받는다.
if not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

%run evaluation/run_generation_eval.py

## 6. 결과 확인 + **Drive 백업** (세션 닫기 전에 반드시)

- `result/ragas_scores.csv` — answerable·비기권 답변의 seed별 RAGAS 점수(faithfulness / answer_relevancy, 0~1)
- `result/generations_eval.jsonl` — 생성 답변 원본 431건

`/content` 는 세션이 끝나면 통째로 사라진다. 아래 셀이 두 파일을 Drive(`MyDrive/rag_eval_result/`)로 복사한다.

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

COLS = ['faithfulness', 'answer_relevancy']
ragas = pd.read_csv('result/ragas_scores.csv')
print('판정 건수:', len(ragas), '| 점수 누락:', int(ragas[COLS].isna().sum().sum()))
display(ragas.groupby('seed')[COLS].mean().round(4))
display(ragas.head(20))

# ── Drive 백업 ──────────────────────────────────────────────
DEST = Path('/content/drive/MyDrive/rag_eval_result')
if Path('/content/drive/MyDrive').exists():        # 1-1 셀에서 마운트했으면 존재
    DEST.mkdir(parents=True, exist_ok=True)
    for f in ('result/ragas_scores.csv', 'result/generations_eval.jsonl'):
        if Path(f).exists():
            shutil.copy(f, DEST / Path(f).name)
            print('백업:', DEST / Path(f).name)
else:
    print('Drive 미마운트 — 1-1 셀을 먼저 실행하거나 좌측 파일탭에서 직접 내려받을 것')

## 7. (E2) 리트리버 파인튜닝 전/후 비교 — seed 3개 평균±표준편차

**4·5번과 독립적** — 새 세션에서 0~3번(설치)만 하고 바로 이 셀을 돌려도 된다.
(코퍼스를 HF 스트리밍으로 자체 수집하고, 모델도 ko-sroberta를 따로 쓴다)

- **SMOKE=1**: 소량(학습 30쿼리 · 네거티브 300)으로 *학습→평가가 끝까지 도는지만* 확인. 몇 분 수준.
- **전량**: E1과 같은 20만 규모 코퍼스 + **seed 3회 풀 학습** → 수 시간. GPU 세션 여유를 확인하고,
  파인튜닝 담당과 자원 사용을 조율할 것.
- OpenAI 키 불필요(학습·검색만 함). 결과는 `evaluation/data/finetune_compare_seeds.csv`.


In [ ]:
import os
%cd /content/knowledge-rag-qa

# ── 실행 규모 ────────────────────────────────────────────────
# 주의: 4번 셀에서 SMOKE를 켰다면 여기까지 이어진다. 전량 실행 시 반드시 pop으로 해제할 것.
os.environ['SMOKE'] = '1'          # ← 구조 확인용. 전량 실행 시 이 줄을 지우고 아래 주석 해제
# os.environ.pop('SMOKE', None)    # ← 전량(20만 코퍼스 + seed 3회 학습, 수 시간)

%run evaluation/run_finetune_compare_eval.py